# NOAA Raster Clipping Workflow

Clips the selected beach NOAA GeoTIFF rasters to the configured study polygon, writes clipped rasters, and saves an area/elevation summary CSV. Before running, set `BEACH` and confirm the polygon coordinates for that beach.

In [ ]:
# Manual choice: set BEACH to "bass" or "bicentennial" before running.
# Confirm POLYGON_LONLAT before regenerating clipped rasters.

from pathlib import Path
import csv
import rasterio
import numpy as np
from rasterio import mask
from shapely.geometry import Polygon, mapping
from pyproj import Transformer


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Codes").is_dir() and (candidate / "Datasets").is_dir() and (candidate / "Models").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing Codes, Datasets, and Models")


PROJECT_ROOT = find_project_root()
DATASETS = PROJECT_ROOT / "Datasets"

BEACH = "bicentennial"

CONFIG = {
    "bass": {
        "INPUT_FOLDER": DATASETS / "NOAA Original " / "Bass Tif Only",
        "OUTPUT_FOLDER": DATASETS / "BCLIPPED",
        # Lon/lat corners derived from the current Bass clipped-raster extent.
        # Replace with a tighter study polygon if you need to regenerate products exactly.
        "POLYGON_LONLAT": [
            (-81.0849047552, 29.3887015773),
            (-81.0800061665, 29.3887046070),
            (-81.0799983770, 29.3787467785),
            (-81.0848964887, 29.3787437500),
        ],
        "AREA_CSV": "area_results_bass_ft.csv",
    },
    "bicentennial": {
        "INPUT_FOLDER": DATASETS / "NOAA Original " / "Bicent Tif Only",
        "OUTPUT_FOLDER": DATASETS / "BICLIPPED",
        # Lon/lat corners in order TL, TR, BR, BL.
        "POLYGON_LONLAT": [
            (-81.0621, 29.3404),
            (-81.0617, 29.3404),
            (-81.0590, 29.3344),
            (-81.0593, 29.3344),
        ],
        "AREA_CSV": "area_results_bicentennial_ft.csv",
    },
}

if BEACH not in CONFIG:
    raise ValueError(f"Unknown BEACH {BEACH!r}. Choose one of: {', '.join(CONFIG)}")

cfg = CONFIG[BEACH]
input_folder = cfg["INPUT_FOLDER"]
output_folder = cfg["OUTPUT_FOLDER"]
output_folder.mkdir(parents=True, exist_ok=True)

for label, path in [("INPUT_FOLDER", input_folder), ("OUTPUT_FOLDER", output_folder)]:
    if not path.exists():
        raise FileNotFoundError(f"{label} does not exist: {path}")

# Transform lon/lat -> Florida East (EPSG:6438, US survey feet)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:6438", always_xy=True)
coords_6438 = [transformer.transform(lon, lat) for lon, lat in cfg["POLYGON_LONLAT"]]
polygon = Polygon(coords_6438)

csv_out = output_folder / cfg["AREA_CSV"]

print("Project root:", PROJECT_ROOT)
print("Beach:", BEACH)
print("Input folder:", input_folder)
print("Output folder:", output_folder)
print("Area CSV:", csv_out)

with open(csv_out, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "area_ft2", "pixel_count", "mean_elev_ft", "min_elev_ft", "max_elev_ft"])

    for src_path in sorted(input_folder.glob("*.tif")):
        if src_path.name.lower().endswith("_uncert.tif"):
            continue

        print(f"Processing {src_path.name} ...")

        with rasterio.open(src_path) as src:
            try:
                out_image, out_transform = mask.mask(
                    src, [mapping(polygon)], crop=True, filled=True, nodata=np.nan
                )
            except Exception as e:
                print(f"Skipping {src_path.name}: {e}")
                continue

            data = out_image[0]
            valid = data[~np.isnan(data)]
            pixel_count = valid.size

            if pixel_count == 0:
                print(f"No overlap with polygon in {src_path.name}")
                continue

            px, py = src.res
            pixel_area_ft2 = abs(px * py)
            area_ft2 = pixel_count * pixel_area_ft2

            mean_elev = float(np.nanmean(valid))
            min_elev = float(np.nanmin(valid))
            max_elev = float(np.nanmax(valid))

            writer.writerow([src_path.name, area_ft2, pixel_count, mean_elev, min_elev, max_elev])

            profile = src.profile.copy()
            profile.update({
                "height": data.shape[0],
                "width": data.shape[1],
                "transform": out_transform,
                "nodata": np.nan,
            })

            out_path = output_folder / src_path.name.replace(".tif", "_clipped.tif")
            with rasterio.open(out_path, "w", **profile) as dst:
                dst.write(out_image)

            print(f"Saved clipped raster -> {out_path.name}")

print(f"Completed. Area results saved to {csv_out}")
